In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import savemat
from wfdb.io import rdsamp

----

In [2]:
crsidlab_root = Path('/home/thulio/projects/masters-research/CRSIDLab_Setembro_2024')

In [3]:
csvs_root = Path('/home/thulio/projects/masters-research/csvs')

In [4]:
dataset_root = Path('/home/thulio/projects/masters-research/data')

In [5]:
relevant_signals = ['II', 'PLETH', 'ABP']

In [7]:
df_samples = pd.read_csv(csvs_root / 'samples.csv')
df_samples

,subject_id,master_id,segment_id,class,sample_start,sample_end
0,48480,p048480-2116-05-04-17-50,3876369_0016,control,3336416,3373916
1,63646,p063646-2177-05-29-11-39,3089087_0007,control,408125,445625
2,64485,p064485-2168-10-15-13-24,3625999_0008,control,3275317,3312817
3,64916,p064916-2102-02-22-16-40,3181603_0003,control,2127014,2164514
4,66682,p066682-2130-05-10-12-40,3549618_0015,control,4263513,4301013
5,67763,p067763-2163-09-18-02-36,3295996_0009,control,225961,263461
6,69006,p069006-2174-02-05-18-35,3098482_0012,control,3456439,3493939
7,76178,p076178-2109-12-20-20-52,3045819_0010,control,1201238,1238738
8,76237,p076237-2128-09-16-10-40,3670140_0003,control,439315,476815
9,81633,p081633-2118-12-25-07-43,3588494_0011,control,3868948,3906448


In [ ]:
columns = ['subject_id', 'segment_id', 'class', 'sample_start', 'sample_end']
for subject_id, segment_id, label, sample_start, sample_end in df_samples[columns].itertuples(index=False):
    p_folder: Path = dataset_root / label / f'p{subject_id:06d}'
    signals, fields = rdsamp(p_folder / segment_id, sampfrom=sample_start, sampto=sample_end, channel_names=relevant_signals)
    patient_file = crsidlab_root / 'subjects' / f'{p_folder.name}_{segment_id}.mat'
    mat_data = {
        'fs': fields['fs'],
        'timeVec': np.arange(fields['sig_len']) / fields['fs'],
        'ecg': signals[:, 0],
        'pleth': signals[:, 1],
        'abp': signals[:, 2],
        # 'resp': signals[:, 3],
    }
    savemat(patient_file, mat_data)